# 🏥 Painel Executivo de Suporte à Decisão — SESPA / Pará
### Projeto Integrador: Da Ingestão à Decisão

---

## 🎯 1. A Pergunta de Decisão
> *"Cruzando as bases do **INPE (focos de calor)**, **OpenDataSUS (SRAG hospitalizado)** e **IBGE (população do Censo 2022)**, identificamos que o risco de sobrecarga por SRAG no mês seguinte é previsto principalmente pelo histórico recente de casos de cada município, e os focos de calor não agregaram poder preditivo neste recorte (Pará, 2023). Recomendamos que a **Secretaria de Estado de Saúde Pública do Pará (SESPA)** envie **kits preventivos de insumos respiratórios** nos próximos **30 dias**, priorizando os municípios com score acima do limiar adotado. Se agir, o ganho esperado é reduzir o custo esperado frente a não agir e frente a alertar todos os municípios (valores na seção 3); se errarmos, cada alerta falso custa **R$ 3.500** e cada município com sobrecarga não alertado custa **R$ 45.000** (premissas do grupo, sem fonte pública)."*

* **Decisor:** Secretaria de Estado de Saúde Pública do Pará (SESPA)
* **Ação Concreta:** Distribuição antecipada de cilindros de oxigênio e broncodilatadores nos hospitais regionais de referência
* **Camada Consumida:** Camada Gold (`data/gold/`)
* **Relatório completo:** `docs/requisito_ml_ready_decisao.md` (comparação de estratégias, sensibilidade aos custos e limitações)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurações visuais
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

# Carregar dados da Camada Gold
df_gold_ind = pd.read_parquet("../data/gold/gold_painel_indicadores_pa.parquet")
df_decisao = pd.read_parquet("../data/gold/gold_recomendacao_decisao_sespa.parquet")

print(f"Painel Gold de Indicadores: {df_gold_ind.shape[0]} registros")
print(f"Tabela de Decisão SESPA: {df_decisao.shape[0]} registros no período operacional")

## 📈 2. Análise da Dinâmica: Queimadas vs. Casos de SRAG Hospitalizados no Pará
Abaixo está o comportamento temporal agregado no estado ao longo dos 12 meses de 2023. O gráfico mostra as duas séries lado a lado e **não demonstra relação causal**: a correlação entre focos e casos por município-mês é impressa logo abaixo do gráfico.

In [ ]:
# Agregação mensal estadual
mensal_pa = df_gold_ind.groupby("ano_mes").agg(
    total_focos=("total_focos_calor", "sum"),
    total_internacoes=("total_internacoes", "sum"),
    taxa_media_100k=("taxa_internacao_100k", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(13, 6))

cor_focos = "#d95f02"
cor_intern = "#1f78b4"

ax1.set_title("Evolução Temporal no Pará (2023): Focos de Queimada (INPE) vs. SRAG Hospitalizado (OpenDataSUS)", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Mês de Competência", fontweight="bold")
ax1.set_ylabel("Total de Focos de Calor (INPE)", color=cor_focos, fontweight="bold")
bars = ax1.bar(mensal_pa["ano_mes"], mensal_pa["total_focos"], color=cor_focos, alpha=0.6, width=0.5, label="Focos de Queimada")
ax1.tick_params(axis="y", labelcolor=cor_focos)

ax2 = ax1.twinx()
ax2.set_ylabel("Taxa Média de SRAG Hospitalizado / 100k hab", color=cor_intern, fontweight="bold")
line = ax2.plot(mensal_pa["ano_mes"], mensal_pa["taxa_media_100k"], color=cor_intern, linewidth=3, marker="o", markersize=8, label="Taxa de SRAG hospitalizado")
ax2.tick_params(axis="y", labelcolor=cor_intern)

# Destacar período de estiagem
ax1.axvspan(6.5, 10.5, color="yellow", alpha=0.15, label="Período de Estiagem (Jul-Nov)")

fig.tight_layout()
plt.show()

r = df_gold_ind["total_focos_calor"].corr(df_gold_ind["total_internacoes"])
print(f"Correlação de Pearson entre focos de calor e casos de SRAG (mesmo município-mês): {r:.3f}")

## 🛡️ 3. Desempenho do Modelo e Comparação com Referências
Comparativo entre estratégias no período de teste (Setembro a Novembro de 2023), todas avaliadas pelo **custo esperado** com a mesma matriz de custos. O score do Random Forest **não é uma probabilidade calibrada**; o limiar foi escolhido por menor custo em validação cronológica **dentro do treino**. As demais comparações (persistência, modelo só com população, sem focos de calor) e a sensibilidade aos custos estão em `docs/requisito_ml_ready_decisao.md`.

In [ ]:
CUSTO_FP, CUSTO_FN = 3500.0, 45000.0   # premissas do grupo, sem fonte pública
y = df_decisao["label_sobrecarga_proximo_mes"].astype(int).to_numpy().astype(bool)
limiar = float(df_decisao["limiar_adotado"].iloc[0])

def avalia(nome, pred):
    pred = np.asarray(pred).astype(bool)
    tp = int((pred & y).sum()); fp = int((pred & ~y).sum()); fn = int((~pred & y).sum())
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f2 = 5 * prec * rec / (4 * prec + rec) if (4 * prec + rec) else 0.0
    pt = lambda v: f"{v:.1%}".replace(".", ",")
    return {
        "Estratégia": nome, "Alertas": tp + fp,
        "Recall": pt(rec), "Precisão": pt(prec), "F2": pt(f2),
        "Custo esperado (R$)": f"{fp * CUSTO_FP + fn * CUSTO_FN:,.0f}".replace(",", "."),
    }

df_metricas = pd.DataFrame([
    avalia("Não alertar ninguém", np.zeros(len(df_decisao))),
    avalia("Alertar todos", np.ones(len(df_decisao))),
    avalia("Regra: foco de calor na estiagem", (df_decisao["total_focos_calor"] > 0) & (df_decisao["periodo_estiagem"] == 1)),
    avalia(f"Random Forest (limiar {limiar:.2f})", df_decisao["probabilidade_risco"] >= limiar),
])
display(df_metricas)

print("-" * 70)
print("Trade-off Econômico-Sanitário da Decisão (premissas do grupo, sem fonte pública):")
fmt = lambda v: f"{v:,.0f}".replace(",", ".")
razao = f"{CUSTO_FN / CUSTO_FP:.1f}".replace(".", ",")
print(f"• Custo do Falso Positivo (FP): R$ {fmt(CUSTO_FP)} (remanejamento de insumos estocáveis)")
print(f"• Custo do Falso Negativo (FN): R$ {fmt(CUSTO_FN)} (resgate, leito intensivo e risco de vida)")
print(f"• Razão: um falso negativo custa ~{razao} vezes um falso positivo.")
print(f"• Positivos no teste: {int(y.sum())} de {len(y)} município-mês; com amostra tão pequena, a evidência é modesta.")

## 📋 4. Tabela de Recomendação e Priorização de Municípios (último mês do teste)
Abaixo está a **tabela prescritiva** ordenada por score, com a lista priorizada de municípios para envio de insumos respiratórios. O score ordena os municípios; **não deve ser lido como probabilidade real**.

In [ ]:
# Filtrar para o último mês do período de teste
mes_ref = df_decisao["ano_mes_t0"].max()
decisao_ref = df_decisao[df_decisao["ano_mes_t0"] == mes_ref].copy()
ranking_top = decisao_ref.sort_values(by="probabilidade_risco", ascending=False).head(15)
print(f"Mês de referência: {mes_ref} | municípios com score >= limiar adotado ({limiar:.2f}): {int((decisao_ref['probabilidade_risco'] >= limiar).sum())} de {len(decisao_ref)}")

cols_apresentacao = [
    "codigo_ibge", "nome_municipio", "populacao_censo_2022",
    "total_focos_calor", "taxa_internacao_100k", "probabilidade_risco", "decisao_recomendada"
]

display(ranking_top[cols_apresentacao])

In [ ]:
# Visualização gráfica do ranking de risco
alto = max(0.70, limiar)
plt.figure(figsize=(12, 7))
palette = ["#d95f02" if r >= alto else "#e6ab02" for r in ranking_top["probabilidade_risco"]]

sns.barplot(
    data=ranking_top,
    x="probabilidade_risco",
    y="nome_municipio",
    palette=palette
)

plt.axvline(alto, color="red", linestyle="--", label=f"Alerta Alto (score >= {alto:.2f})")
plt.axvline(limiar, color="orange", linestyle=":", label=f"Limiar adotado (score >= {limiar:.2f})")

plt.title("Top 15 Municípios do Pará Priorizados para Alocação de Kits de Emergência (SESPA)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Score do modelo (não é probabilidade calibrada)", fontweight="bold")
plt.ylabel("Município", fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 🔍 5. Limitações e Próximos Passos
1. **O que os dados permitem afirmar:** no Pará, em 2023, o histórico recente de casos de SRAG de cada município prevê a sobrecarga do mês seguinte melhor do que as regras simples comparadas. Os focos de calor **não agregaram poder preditivo mensurável** (veja a correlação impressa na seção 2 e a comparação "com e sem focos" no relatório). O modelo completo foi o de menor custo no teste, mas com poucos positivos e dependendo da premissa de custo.
2. **Limitações:** a base é o SIVEP-Gripe (SRAG): casos notificados de síndrome respiratória aguda grave, hospitalizados ou óbitos, e não todas as internações respiratórias; está sujeita a atraso e subnotificação. Os dados cobrem um único ano, o teste tem 3 meses, os casos são atribuídos ao município de residência e os custos de FP e FN são premissas do grupo.
3. **Próximos passos recomendados:** incorporar dados de qualidade do ar (material particulado PM2.5 e PM10) e de vento e chuva do INMET; usar vários anos (INPE e SIVEP-Gripe têm séries desde 2019) para treinar com ciclos sazonais completos; e obter os custos reais com a SESPA.